# Telco Churn: comparacion de modelos de clasificacion


- Carga de datos Telco Churn desde URL
- Preprocesamiento
- Entrenamiento de multiples modelos: AdaBoost, Random Forest, KNN, DT, Regresion Logistica, CatBoost, XGBoost, GradientBoosting y LightGBM
- Matriz de confusion y metricas (accuracy, precision, recall, F1, AUC, etc.)
- Curva ROC y AUC por modelo
- Interpretacion final


> Nota: algunas librerias (CatBoost, XGBoost, LightGBM) son opcionales. Si no estan instaladas, el notebook continua con los modelos disponibles.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    balanced_accuracy_score,
)

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
""""
!pip install catboost
!pip install xgboost
!pip install lightgbm
""""""

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 1.0/100.2 MB 7.1 MB/s eta 0:00:14
   - -------------------------------------- 3.4/100.2 MB 9.6 MB/s eta 0:00:11
   -- ------------------------------------- 6.0/100.2 MB 10.5 MB/s eta 0:00:09
   --- ------------------------------------ 8.4/100.2 MB 10.8 MB/s eta 0:00:09
   --- ------------------------------------ 9.7/100.2 MB 9.7 MB/s eta 0:00:10
   ---- ----------------------------------- 12.3/100.2 MB 10.2 MB/s eta 0:00:09
   ----- ---------------------------------- 14.7/100.2 MB 10.4 MB/s eta 0:00:09
   ------ --------------------------------- 17.3/100.2 MB 10.6 MB/s eta 0:00:08
   ------- -------------------------------- 19.7/100.2 MB 10.8 MB/s eta 0:00:08
   -------- ------------------------------- 22.3/100.2 MB 10.8 MB/s eta 0:00:08
   --------- ------------------------------ 24.6/100.2 MB 11.0 MB/s eta 0:00:07
   ---------- ----------------------------- 27.3/100.2 MB


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 8.2 MB/s eta 0:00:13
   - -------------------------------------- 2.9/101.7 MB 11.2 MB/s eta 0:00:09
   -- ------------------------------------- 5.5/101.7 MB 11.5 MB/s eta 0:00:09
   --- ------------------------------------ 7.9/101.7 MB 11.6 MB/s eta 0:00:09
   ---- ----------------------------------- 10.5/101.7 MB 11.7 MB/s eta 0:00:08
   ----- ---------------------------------- 13.1/101.7 MB 11.7 MB/s eta 0:00:08
   ------ --------------------------------- 15.7/101.7 MB 11.8 MB/s eta 0:00:08
   ------- -------------------------------- 18.1/101.7 MB 11.8 MB/s eta 0:00:08
   -------- ------------------------------- 20.7/101.7 MB 11.8 MB/s eta 0:00:07
   --------- ------------------------------ 23.3/101.7 MB 11.7 MB/s eta 0:00:07
   ---------- ----------------------------- 25.7/101.7 MB 11.8 MB/s eta 0:00:07
   ----------- ---------------------------- 28.3/101.7


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 5.6 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 4.0 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#pip install --upgrade pip

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------- ----- 1.6/1.8 MB 12.0 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 7.1 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.0.1
    Uninstalling pip-26.0.1:
      Successfully uninstalled pip-26.0.1
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Importacion opcional de modelos externos
catboost_disponible = True
xgboost_disponible = True
lightgbm_disponible = True

try:
    from catboost import CatBoostClassifier
except Exception:
    catboost_disponible = False

try:
    from xgboost import XGBClassifier
except Exception:
    xgboost_disponible = False

try:
    from lightgbm import LGBMClassifier
except Exception:
    lightgbm_disponible = False

print('CatBoost disponible:', catboost_disponible)
print('XGBoost disponible:', xgboost_disponible)
print('LightGBM disponible:', lightgbm_disponible)

## 1) Cargar Telco Churn desde URL

In [ ]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
print('Shape:', df.shape)
df.head()

## 2) Limpieza y preprocesamiento

- `TotalCharges` viene como texto en algunos casos y se convierte a numerico.
- Se elimina `customerID` por ser un identificador.
- La variable objetivo `Churn` se pasa a 0/1.

In [ ]:
# Copia para no modificar el original
data = df.copy()

# Convertir TotalCharges a numerico
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

# Eliminar nulos que salgan de la conversion
data = data.dropna().reset_index(drop=True)

# Eliminar ID
if 'customerID' in data.columns:
    data = data.drop(columns=['customerID'])

# Target a 0/1
data['Churn'] = data['Churn'].map({'No': 0, 'Yes': 1})

print('Shape despues de limpieza:', data.shape)
data.head()

In [ ]:
# Distribucion de clases (prevalencia)
conteo = data['Churn'].value_counts().sort_index()
prevalencia = data['Churn'].value_counts(normalize=True).sort_index()

resumen_clases = pd.DataFrame({
    'conteo': conteo,
    'proporcion': prevalencia
})
resumen_clases.index = ['Clase negativa (0: No churn)', 'Clase positiva (1: Churn)']

print('Prevalencia de clases:')
display(resumen_clases)

plt.figure(figsize=(5,4))
sns.countplot(x=data['Churn'])
plt.title('Distribucion de clases (0=No Churn, 1=Churn)')
plt.show()

## 3) Separar X/y y dividir train-test

In [ ]:
X = data.drop(columns=['Churn'])
y = data['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# Columnas numericas y categoricas
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print('Numericas:', len(numeric_features))
print('Categoricas:', len(categorical_features))

# Preprocesador: escalar numericas + one-hot categoricas
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

## 4) Definir modelos

In [ ]:
modelos = {
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=300),
    'KNN': KNeighborsClassifier(n_neighbors=15),
    'Decision Tree (DT)': DecisionTreeClassifier(random_state=42),
    'Regresion Logistica': LogisticRegression(max_iter=2000, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
}

if catboost_disponible:
    modelos['CatBoost'] = CatBoostClassifier(verbose=0, random_state=42)

if xgboost_disponible:
    modelos['XGBoost'] = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=42,
    )

if lightgbm_disponible:
    modelos['LightGBM'] = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
    )

print('Modelos a entrenar:')
for m in modelos:
    print('-', m)

## 5) Entrenamiento y metricas

Metricas incluidas:
- Accuracy
- Precision clase positiva (1)
- Precision clase negativa (0)
- Recall (sensibilidad)
- Specificity (recall de clase negativa)
- F1 score
- Balanced Accuracy
- ROC AUC

In [ ]:
resultados = []
conf_matrices = {}
roc_curvas = {}

for nombre, modelo in modelos.items():
    pipe = Pipeline(steps=[
        ('prep', preprocess),
        ('model', modelo)
    ])

    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    # Probabilidades para ROC/AUC
    if hasattr(pipe, 'predict_proba'):
        y_proba = pipe.predict_proba(X_test)[:, 1]
    else:
        # fallback por si algun modelo no tuviera predict_proba
        scores = pipe.decision_function(X_test)
        y_proba = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)

    cm = confusion_matrix(y_test, y_pred)
    conf_matrices[nombre] = cm

    tn, fp, fn, tp = cm.ravel()

    # Precision de clase positiva y negativa
    precision_pos = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    precision_neg = precision_score(y_test, y_pred, pos_label=0, zero_division=0)

    # Recall clase positiva (sensibilidad) y clase negativa (specificity)
    recall_pos = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    specificity = recall_score(y_test, y_pred, pos_label=0, zero_division=0)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_modelo = roc_auc_score(y_test, y_proba)
    roc_curvas[nombre] = (fpr, tpr, auc_modelo)

    resultados.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision_positiva': precision_pos,
        'Precision_negativa': precision_neg,
        'Recall_positivo': recall_pos,
        'Specificity_negativa': specificity,
        'F1_score': f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred),
        'ROC_AUC': auc_modelo,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
    })

resultados_df = pd.DataFrame(resultados).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
resultados_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision_positiva': '{:.4f}',
    'Precision_negativa': '{:.4f}',
    'Recall_positivo': '{:.4f}',
    'Specificity_negativa': '{:.4f}',
    'F1_score': '{:.4f}',
    'Balanced_Accuracy': '{:.4f}',
    'ROC_AUC': '{:.4f}',
})

## 6) Matriz de confusion por modelo

In [ ]:
n_modelos = len(conf_matrices)
cols = 3
rows = int(np.ceil(n_modelos / cols))

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)

for i, (nombre, cm) in enumerate(conf_matrices.items()):
    ax = axes[i]
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn (0)', 'Churn (1)'])
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(nombre)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Matrices de confusion por modelo', fontsize=14)
plt.tight_layout()
plt.show()

## 7) Curvas ROC y AUC por modelo

In [ ]:
plt.figure(figsize=(9, 7))

for nombre, (fpr, tpr, auc_modelo) in roc_curvas.items():
    plt.plot(fpr, tpr, linewidth=2, label=f'{nombre} (AUC={auc_modelo:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio (AUC=0.500)')
plt.xlabel('FPR - Tasa de falsos positivos')
plt.ylabel('TPR - Tasa de verdaderos positivos')
plt.title('Curvas ROC comparativas')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.show()

## 8) Conclusiones para clase

- El mejor modelo depende de la metrica objetivo (AUC, F1, recall, etc.).
- Si importa no perder churners, prioriza **Recall positivo**.
- Si importa reducir falsas alarmas, revisa **Precision positiva** y **Specificity**.
- El AUC ayuda a comparar modelos de forma global sin fijar un umbral unico.

In [ ]:
top_auc = resultados_df.iloc[0]
print('Mejor modelo por ROC_AUC:', top_auc['Modelo'])
print(f"AUC: {top_auc['ROC_AUC']:.4f}")
print(f"F1: {top_auc['F1_score']:.4f}")
print(f"Recall positivo: {top_auc['Recall_positivo']:.4f}")
print(f"Precision positiva: {top_auc['Precision_positiva']:.4f}")